# Stage 4 — Text Splitting

**Project:** ResearchMate — Research Paper RAG Chatbot
**Goal of this notebook:** Decide a chunk size based on the *actual* measured length of our documents (not a guess), run LangChain's text splitter, and verify that abstracts stay intact as single chunks.

**Before running:** upload `train_prepared.csv` (from Stage 2) to this Colab session.

## Cell 1 — Install packages

`langchain-text-splitters` for the splitter, `langchain-core` for the `Document` class (same as Stage 3).

In [1]:
!pip install -q langchain-text-splitters langchain-core

## Cell 2 — Reload data and rebuild the Document list

Same steps as Stage 3: load the prepared CSV and rebuild the list of `Document` objects, since each notebook stage is self-contained.

In [2]:
import pandas as pd
from langchain_core.documents import Document

df = pd.read_csv("train_prepared.csv")

documents = []
for _, row in df.iterrows():
    doc = Document(
        page_content=row["text"],
        metadata={
            "id": row["ID"],
            "title": row["TITLE"],
            "topics": row["topics"],
        }
    )
    documents.append(doc)

print("Total documents:", len(documents))

Total documents: 20971


## Cell 3 — Measure actual document lengths (in characters)

Text splitters work in characters, not words, so we measure character length directly rather than reusing the word-count stats from Stage 1. This tells us exactly how big our chunk size needs to be to avoid splitting abstracts unnecessarily.

In [3]:
char_lengths = pd.Series([len(doc.page_content) for doc in documents])

print(char_lengths.describe())
print("\nMax document length (characters):", int(char_lengths.max()))

count    20971.000000
mean      1084.050737
std        414.548141
min         77.000000
25%        784.000000
50%       1065.000000
75%       1368.000000
max       2862.000000
dtype: float64

Max document length (characters): 2862


## Cell 4 — Choose chunk_size and chunk_overlap based on the measured max length

We set `chunk_size` comfortably above the true maximum document length (rounded up, plus a buffer), so the splitter structurally cannot break any abstract into multiple pieces. `chunk_overlap` is kept small, purely as a safety margin.

In [4]:
max_length = int(char_lengths.max())

# Round up to the next 500, then add a buffer of 500 characters
chunk_size = ((max_length // 500) + 1) * 500 + 500
chunk_overlap = 200

print(f"Measured max document length: {max_length} characters")
print(f"Chosen chunk_size: {chunk_size}")
print(f"Chosen chunk_overlap: {chunk_overlap}")

Measured max document length: 2862 characters
Chosen chunk_size: 3500
Chosen chunk_overlap: 200


## Cell 5 — Split the documents

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first (paragraphs, then sentences, then words) only if a document exceeds `chunk_size`. Since we set `chunk_size` above the true maximum length, we expect it to leave every document as a single, unsplit chunk — but we verify this in the next cell rather than assuming it.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

split_docs = splitter.split_documents(documents)

print("Documents before splitting:", len(documents))
print("Chunks after splitting:", len(split_docs))

Documents before splitting: 20971
Chunks after splitting: 20971


## Cell 6 — Verify: did any abstract actually get split?

If splitting worked as expected, the number of chunks should equal the number of original documents (1 chunk per paper). We check this directly by counting how many times each paper's `id` appears among the chunks — any `id` appearing more than once means that paper was split into multiple chunks.

In [6]:
from collections import Counter

id_counts = Counter(doc.metadata["id"] for doc in split_docs)
split_papers = {paper_id: count for paper_id, count in id_counts.items() if count > 1}

print("Papers that were split into more than one chunk:", len(split_papers))

if split_papers:
    example_id = list(split_papers.keys())[0]
    print("\nExample split paper id:", example_id)
    for doc in split_docs:
        if doc.metadata["id"] == example_id:
            print("-" * 40)
            print(doc.page_content[:300], "...")

Papers that were split into more than one chunk: 0


## What to check after running this notebook

- **Cell 3:** note the measured max character length — this is the real number driving our chunk-size decision (not a guess).
- **Cell 4:** confirm the chosen `chunk_size` is comfortably above that max.
- **Cell 5:** compare the "documents before" vs "chunks after" counts — they should be equal or very close.
- **Cell 6:** ideally **0 papers** were split into multiple chunks, confirming each abstract stayed intact as one coherent chunk. If a handful were split, that's fine too — just note how many and glance at the example.

Paste back the max length from Cell 3, the chosen chunk_size from Cell 4, the before/after counts from Cell 5, and the split-papers count from Cell 6 — then we'll move to Stage 5 (Embeddings).